# Setup

In [270]:
from pathlib import Path
import os
import subprocess
import shutil
import time
import pandas as pd
import sys

In [271]:
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["OPENBLAS_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"
os.environ["VECLIB_MAXIMUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"

In [272]:
FSLDIR = "/mnt/1248/open/usr/local/fsl"
fsl_setup = subprocess.run(
    ["bash", "-c", f"source {FSLDIR}/etc/fslconf/fsl.sh && env"],
    check=True,
    capture_output=True,
    text=True,
)

for line in fsl_setup.stdout.splitlines():
    key, separator, value = line.partition("=")
    if separator:
        os.environ[key] = value

os.environ["FSLDIR"] = FSLDIR
os.environ["PATH"] = f"{FSLDIR}/bin:{os.environ['PATH']}"

print(f"FSLDIR={os.environ['FSLDIR']}")
print(f"fslroi={shutil.which('fslroi')}")

FSLDIR=/mnt/1248/open/usr/local/fsl
fslroi=/mnt/1248/open/usr/local/fsl/bin/fslroi


# Tools
Note: All functions (except d2n) input patient folder, then everything will auto sort.

## D2N
Includes special sorting for dataset5 folder structure. Does not input the usual patient folder path.

Inputs patient number path without the _date. Example: /mnt/1248/open/dataset5/gLymphVIDA/Hydro_1/0662719

Cannot chain run.

In [273]:
def d2n(d_patient_folderpath: str, n_group_folderpath: str, silent = False, **kwargs):
    
    patientid = Path(d_patient_folderpath).name
    input_folderpath = Path(d_patient_folderpath)

    if not silent:
        print(f"Searching {input_folderpath} for MRI folders.")
        print(f"Found {sum(1 for f in Path().iterdir() if f.is_dir())} folders in {patientid}.")

    

    giveup = []

    for folder in input_folderpath.iterdir():
        if Path(f"{folder}/MRI").exists():

            flag_count = 0
            date = folder.name
            output_folderpath = Path(n_group_folderpath) / f"{patientid}_{date}" / "raw"
            !mkdir -p {output_folderpath}

            while not any(output_folderpath.iterdir()): # Check if the folder got the nii files, if not then flag and restart.
                if not silent:
                    print(f"Running dcm2niix for {patientid}_{date}.")
                dcm_folder = folder / "MRI" / "dDTI_b1000_d30"
                if Path(dcm_folder).exists():
                    # cmd = [
                    #     "dcm2niix",
                    #     "-z", "y",
                    #     "-o", str(output_folderpath),
                    #     str(dcm_folder),
                    # ]
                    # subprocess.run(cmd, capture_output=True, text=True, check=True)
                    if len(list(Path(dcm_folder).iterdir())) >= 300:
                        print("Too many files. Skipping.")
                        giveup.append(f"{patientid}_{date}")
                        break

                    # for file in Path(dcm_folder).iterdir():
                    #     print(file.name)
                    

                    !dcm2niix -z y -o {output_folderpath} {dcm_folder}


                    if not any(output_folderpath.iterdir()):
                        flag_count += 1
                        print(f"Output not found. ({flag_count})")
                    if flag_count == 3:
                        print(f"Flagged 3 times, skipping {patientid}_{date}.")
                        giveup.append(f"{patientid}_{date}")
                        break
                    if any("DeIdentified" in f.name for f in Path(output_folderpath).iterdir()):
                        print(f"Removed D2N result for patient \033[4m{patientid}\033[0m because deidentified.")
                        giveup.append(f"{patientid}_{date}")
                        break
                    
                else: 
                    print("Folder format weird.")
                    giveup.append(f"{patientid}_{date}")
                    break

    return {"d2n_empty_folders": giveup}

## BET

In [274]:
def bet(patient_folderpath: str, silent = False, **kwargs):

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    path_raw = Path(patient_folderpath) / "raw"
    niigz_files = list(path_raw.glob('*.nii.gz'))

    if Path(f"{folderpath}/bet/{patientid}_b0_R_mask.nii.gz").exists() == True:
        if not silent:
            print(f"Mask for {patientid} already exists. Skipping BET.")
        return
    
    if len(niigz_files) != 1:
        print(f"Warning: Found {len(niigz_files)} .nii.gz files in {path_raw}.")
    if not niigz_files:
        raise FileNotFoundError(f"No .nii.gz file found in {path_raw}")

    niigz_file = niigz_files[0].name
    if not silent:
        print(f"Selected {niigz_file} for mask generation.")

    !mkdir -p {folderpath}/bet

    !fslroi {path_raw}/{niigz_file} {folderpath}/bet/{patientid}_b0.nii.gz 0 1

    !bet {folderpath}/bet/{patientid}_b0.nii.gz {folderpath}/bet/{patientid}_b0_R.nii.gz -m -f 0.3 -R

    !bet {folderpath}/bet/{patientid}_b0.nii.gz {folderpath}/bet/{patientid}_b0_noR.nii.gz -m -f 0.3
    
    # subprocess.run(
    #     [
    #         "bet",
    #         f"{folderpath}/bet/{patientid}_b0.nii.gz",
    #         f"{folderpath}/bet/{patientid}_b0_R.nii.gz",
    #         "-m",
    #         "-f",
    #         "0.3",
    #         "-R",
    #     ],
    #     capture_output=True,
    #     text=True,
    #     check=True,
    # )

    # subprocess.run(
    #     [
    #         "bet",
    #         f"{folderpath}/bet/{patientid}_b0.nii.gz",
    #         f"{folderpath}/bet/{patientid}_b0_noR.nii.gz",
    #         "-m",
    #         "-f",
    #         "0.3",
    #     ],
    #     capture_output=True,
    #     text=True,
    #     check=True,
    # )

    if not silent:
        print(f"Generated masks for {patientid}.")

    !fslmaths {folderpath}/bet/{patientid}_b0_R_mask.nii.gz -sub {folderpath}/bet/{patientid}_b0_noR_mask.nii.gz {folderpath}/bet/diff.nii.gz

    diff = !fslstats {folderpath}/bet/diff.nii.gz -R
    if not silent:
        print("Diff: " + diff[0])
    # if diff[0] != "0.000000 0.000000":
    #     print("The two masks are different. Please check the results.")
    # else:
    #     print("The two masks are the same. Safe to proceed.")

    return None


## EDDY

In [275]:
def eddy(patient_folderpath: str, silent = False, **kwargs):

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    path_raw = Path(patient_folderpath) / "raw"
    niigz_files = list(path_raw.glob('*.nii.gz'))

    if Path(f"{folderpath}/eddy/{patientid}.nii.gz").exists() == True:
        if not silent:
            print(f"EDDY correction for {patientid} already exists. Skipping EDDY.")
        return

    if len(niigz_files) != 1:
        print(f"Warning: Found {len(niigz_files)} .nii.gz files in {path_raw}.")
        # print(niigz_files)
    if not silent:
        print(f"Selected {niigz_files[0].name} for EDDY correction.")
    niigz_file = niigz_files[0].name
    filename_base = niigz_file.replace(".nii.gz", "")
    # print(f"Filename base: {filename_base}")

    !mkdir -p {folderpath}/eddy

    # Make acqparams.txt
    grep = !grep -iE "PhaseEncodingDirection|TotalReadoutTime" "{path_raw}/{filename_base}.json"
    # print(grep)
    trt = grep[0].split(":")[1].strip().strip(",")
    ped = grep[1].split(":")[1].strip().strip("\",")
    
    if ped == "i":
        ped = "1 0 0"
    if ped == "i-":
        ped = "-1 0 0"
    if ped == "j":
        ped = "0 1 0"
    if ped == "j-":
        ped = "0 -1 0"
    if ped == "k":
        ped = "0 0 1"
    if ped == "k-":
        ped = "0 0 -1"

    with open(f"{folderpath}/eddy/acqparams.txt", "w") as f:
        f.write(f"{ped} {trt}")
    if not silent:
        print(f"Created acqparams.txt with {ped} {trt}.")

    # Make index.txt
    n = int(subprocess.check_output(["fslval", f"{path_raw}/{filename_base}.nii.gz", "dim4"], text=True).strip())
    index_line = " ".join(["1"] * n)
    with open(f"{folderpath}/eddy/index.txt", "w") as f:
        f.write(index_line)
    if not silent:
        print(f"Created index.txt with n = {n}.")

    # Run EDDY correction
    if not silent:
        print(f"Running eddy_cuda10.2 for {patientid}...")
    
    subprocess.run(
        [
            "eddy_cuda10.2",
            f"--imain={path_raw}/{filename_base}.nii.gz",
            f"--mask={folderpath}/bet/{patientid}_b0_R_mask.nii.gz",
            f"--acqp={folderpath}/eddy/acqparams.txt",
            f"--index={folderpath}/eddy/index.txt",
            f"--bvecs={path_raw}/{filename_base}.bvec",
            f"--bvals={path_raw}/{filename_base}.bval",
            f"--out={folderpath}/eddy/{patientid}",
            "--repol",
            "--cnr_maps",
            "--data_is_shelled",
            
            # " > /dev/null 2>&1"
        ],
        check = True,
    ) 

    # Copy bval and bvec files to the eddy folder
    bval_file = list(path_raw.glob('*.bval'))[0]
    !cp {bval_file} {folderpath}/eddy/{patientid}.bval

    bvec_file = list(path_raw.glob('*.bvec'))[0]
    !cp {bvec_file} {folderpath}/eddy/{patientid}.bvec

    return None



## SRC

In [276]:
def src(patient_folderpath: str, silent = False, **kwargs) -> None:

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)
    
    path_eddy = Path(patient_folderpath) / "eddy"
    niigz_file = list(path_eddy.glob(f'{patientid}.nii.gz'))[0].name

    if Path(f"{folderpath}/src/{patientid}.sz").exists() == True:
        if not silent:
            print(f"SRC conversion for {patientid} already exists. Skipping SRC.")
        return
    
    if not silent:
        print(f"Selected {niigz_file} for SRC conversion.")

    !mkdir -p {folderpath}/src

    cmd = [
        "dsi_studio",
        "--action=src",
        f"--source={folderpath}/eddy/{patientid}.nii.gz",
        f"--bval={folderpath}/eddy/{patientid}.bval",
        f"--bvec={folderpath}/eddy/{patientid}.bvec",
        f"--output={folderpath}/src/{patientid}",
    ]
    subprocess.run(cmd, capture_output=True, text=True)
    if not silent:
        print(f"Converted {niigz_file} to SRC format.")
    
    return None

## REC

In [277]:
def rec(patient_folderpath: str, silent = False, **kwargs):

    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    path_src = Path(patient_folderpath) / "src"
    src_file = list(path_src.glob(f'{patientid}.sz'))[0].name
    
    if Path(f"{folderpath}/rec/{patientid}.fib.gz").exists() == True:
        if not silent:
            print(f"Reconstruction for {patientid} already exists. Skipping REC.")
        return
    
    if not silent:
        print(f"Selected {src_file} for FIB conversion.")

    !mkdir -p {folderpath}/rec

    cmd = [
        "dsi_studio",
        "--action=rec",
        f"--source={folderpath}/src/{patientid}.sz",
        f"--mask={folderpath}/bet/{patientid}_b0_R_mask.nii.gz",
        f"--method=1",
        f"--other_output=fa,md,ad,rd,gfa,qa,rdi,nrdi",
        f"--output={folderpath}/rec/{patientid}.fib.gz",
    ]
    subprocess.run(cmd, capture_output=True, text=True)

    if not silent:
        print(f"Converted {src_file} to FIB format.")

    return None

## TRK

In [278]:
def trk(patient_folderpath: str, silent = False, **kwargs):
    
    
    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    trk_label = kwargs.get("trk_label", "CC")
    
    if Path(f"{folderpath}/trk/{patientid}_{trk_label}.tt.gz").exists() == True:
        if not silent:
            print(f"Reconstruction for {patientid}_{trk_label} already exists. Skipping TRK.")
        return

    path_rec = Path(patient_folderpath) / "rec"
    # print(path_rec)
    fib_file = list(path_rec.glob(f'{patientid}.fib.gz'))[0].name
    if not silent:
        print(f"Selected {fib_file} for fiber tracking.")

    roi = kwargs.get("trk_roi", ["FreeSurferSeg:CC_Posterior", "FreeSurferSeg:CC_Mid_Posterior", "FreeSurferSeg:CC_Central", "FreeSurferSeg:CC_Mid_Anterior", "FreeSurferSeg:CC_Anterior"])
    if not silent:
        print(roi)
    !mkdir -p {folderpath}/trk

    cmd = [
        "dsi_studio",
        "--action=trk",
        f"--source={folderpath}/rec/{patientid}.fib.gz",
        f"--seed={','.join(roi)}",
        f"--ter={','.join(roi)},dilation,dilation,negate",
        f"--min_length=5",
        f"--tract_count=600000",
        f"--output={folderpath}/trk/{patientid}_{trk_label}.tt.gz",
        f"--export=stat",
        # " > /dev/null 2>&1"
    ]
    subprocess.run(cmd, capture_output=True, text=True)

    if not silent:
        print(f"Successfully tracked fibers for {patientid}_{trk_label}.")

    return

## READSTATS

In [279]:
all_metrics = [
                "number of tracts", "mean length(mm)", "median length(mm)", 
                "span(mm)", "curl", "elongation", "total volume(mm^3)", 
                "LPS quarter volume(mm^3)", "2nd and 3rd quarter volume(mm^3)", 
                "RAI quarter volume(mm^3)", "total surface area(mm^2)", 
                "total radius of end regions(mm)", "total area of end regions(mm^2)",
                "irregularity", "area of LPS end region (mm^2)", "radius of LPS end region (mm)",
                "volume of LPS end branches", "area of RAI end region(mm^2)",
                "radius of RAI end region(mm)", "volume of RAI end branches", 
                "fa", "md", "ad", "rd", "vol", # <---- important stuff maybe idk lol
                "fa of LPS quarter", "md of LPS quarter", "ad of LPS quarter", "rd of LPS quarter",	"vol of LPS quarter", 
                "fa of RAI quarter", "md of RAI quarter", "ad of RAI quarter", "rd of RAI quarter", "vol of RAI quarter",
                ]

In [280]:
def readstats(patient_folderpath: str, silent = False, **kwargs):
    
    patientid = Path(patient_folderpath).name
    folderpath = Path(patient_folderpath)

    trk_label = kwargs.get("trk_label", "CC")
    
    if Path(f"{folderpath}/trk/{patientid}_{trk_label}.tt.gz.stat.txt").exists() == False:
        print("huh trk dont exist?")
        return

    tsv_filepath = Path(patient_folderpath) / "trk" / f"{patientid}_{trk_label}.tt.gz.stat.txt"

    tsv_file = f"{patientid}_{trk_label}.tt.gz.stat.txt"
    if not silent:
        print(f"Selected {tsv_file} for reading.")

    df = pd.read_csv(tsv_filepath, sep='\t', header=None, names=["metric", "value"])
    # print(df)
    met2val = df.set_index("metric")["value"]
    read_metrics = kwargs.get("read_metrics", all_metrics)
    
    output = {}
    for i in read_metrics:
        value = met2val[i]
        output[i] = value

    return output

# Pipeline

In [281]:
def chain(inputpath: str, functions: list, silent: bool = False, **kwargs):
    if Path(inputpath).exists() == False:
        print(f" {inputpath} does not exist.")
        return
    
    chain_output = {}

    for func in functions:
        start = time.perf_counter()
        if not silent:
            print(f" \033[1mRunning {func.__name__.upper()} for {Path(inputpath).name}...\033[0m")
        func_output = func(inputpath, silent=True, **kwargs)
        if func_output != None:
            chain_output[func.__name__] = func_output
        end = time.perf_counter()
        # print(func_output)
        if not silent:
            print(f" Finished {func.__name__.upper()} for {Path(inputpath).name}. ({end - start:.6f}s)\n")

    
    return chain_output

In [282]:
def clean_path(path, server_map={"R730XD": "1248"}):
    if f"/mnt/{server_map['R730XD']}/" in path:
        return path
    parts = path.replace("\\", "/").lstrip("/").split("/")
    return Path("/mnt/" + "/".join([server_map[parts[0].upper()]] + parts[1:]))

In [283]:
# Must have auto skip to avoid redundancy!!!!
def clean_batchd2n_batchchain_collect(groupinputpath: str, groupoutputpath: str, 
                                  functions: list = [bet, eddy, src, rec, trk, readstats], 
                                  silent: bool = False,
                                  collect_filename: str = "data",
                                  **kwargs,
                                  ):

    # Clean
    group_in_path_clean = clean_path(groupinputpath)
    group_out_path_clean = clean_path(groupoutputpath)
    !mkdir -p {group_out_path_clean}

    if Path(group_in_path_clean).exists() == False:
        print(f"{group_in_path_clean} does not exist. Please check again.")
        return

    if not silent:
        print(f"\033[1m\033[4mRunning full batch pipeline...\033[0m\n From:\t{group_in_path_clean}\n To:\t{group_out_path_clean}")

    # Batch D2N
    empty_folders = []
    print("\n\033[4mStarting Batch D2N.\033[0m")
    for d_patient_folder in Path(group_in_path_clean).iterdir():
        if d_patient_folder.is_dir():
            patientid = d_patient_folder.name
            if any(d_patient_folder.iterdir()):
                if any("DeIdentified" in f.name for f in Path(d_patient_folder).iterdir()):
                    None
                    # print(f"Skipped D2N for patient \033[4m{patientid}\033[0m because deidentified.")
                else:
                    print(f"Running D2N for patient \033[4m{patientid}\033[0m...")
                    d2ninfo = d2n(d_patient_folder, group_out_path_clean, silent=True)
                    if d2ninfo["d2n_empty_folders"] != []:
                        for i in d2ninfo["d2n_empty_folders"]:
                            empty_folders.append(i)
            else:
                print(f"Skipped empty folder {patientid}.")
    print(f"Completed batch D2N.")

    if empty_folders != []:
        print(f"Deleting {len(empty_folders)} empty folder(s).")
        for i in empty_folders:
            !rm -R {group_out_path_clean}/{i}

    

    # Batch Chain
    readstats_comp = {}
    lsfunc = [f.__name__ for f in functions]
    print(f"\n\033[4mStarting Batch Chain with {lsfunc}.\033[0m")
    for n_patient_folder in Path(group_out_path_clean).iterdir():
        if n_patient_folder.is_dir():
            patientid_date = n_patient_folder.name

            print(f"Running Chain for patient \033[4m{patientid_date}\033[0m...")
            chaininfo = chain(
                inputpath=n_patient_folder, 
                functions=functions, 
                silent=silent,
                **kwargs,
                )
            sys.stdout.write("\033[F\033[K" * 2)
            readstats_comp[patientid_date] = chaininfo["readstats"] # {patientid_date : readstats}
            

    print("Completed batch Chain.")
    
    # Collect
    if "readstats" in lsfunc:

        df = pd.DataFrame({
            "PatientID": [],
            "Date": [],
        })

        read_metrics = kwargs.get("read_metrics", all_metrics)

        for metric in read_metrics:
            df[metric] = None

        for patient in readstats_comp:
            id, date = patient.split("_")

            new_row = {"PatientID": id, "Date": date}
            df = pd.concat([df, pd.DataFrame([new_row])], ignore_index=True)
            mask = (df["PatientID"] == id) & (df["Date"] == date)
            for metric in read_metrics:
                df.loc[mask, metric] = readstats_comp[patient][metric]
    
        print("\n", df)
        
        df.to_csv(f"{group_out_path_clean}/{collect_filename}.csv", index=False)

        return Path(f"{group_out_path_clean}/{collect_filename}.csv")

    else:
        print("READSTATS not included in functions list. Skipping data collection.")
    
    

# Run

## Full Pipeline Run

In [286]:
# Base
inputpath   : str   = r"\\R730XD\open\dataset5\gLymphVIDA\Hydro_1"
outputpath  : str   = r"\\R730XD\open\finley\hydrocephalus\Hydro_1"
csv_name    : str   = "data"
functions   : list  = [bet, eddy, src, rec, trk, readstats]
silent      : bool  = False

# TRK
trk_label   : str   = "CC"
trk_roi     : list  = ["FreeSurferSeg:CC_Posterior", "FreeSurferSeg:CC_Mid_Posterior", "FreeSurferSeg:CC_Central", "FreeSurferSeg:CC_Mid_Anterior", "FreeSurferSeg:CC_Anterior"]

# READSTATS
read_metrics: list  = ["fa", "md", "ad", "rd", "vol"] # Leave empty to get all metrics

In [287]:
clean_batchd2n_batchchain_collect(
    groupinputpath  = inputpath, 
    groupoutputpath = outputpath,
    collect_filename= csv_name,
    functions       = functions, 
    silent          = silent,
    trk_roi         = trk_roi, 
    trk_label       = trk_label, 
    read_metrics    = read_metrics,
    )

Running full batch pipeline...
 From:	/mnt/1248/open/dataset5/gLymphVIDA/Hydro_1
 To:	/mnt/1248/open/finley/hydrocephalus/Hydro_1

Starting Batch D2N.
Running D2N for patient 5219757...
Running D2N for patient 6202976...
Running D2N for patient 3993426...
Running D2N for patient 7231080...
Running D2N for patient 6036455...
Running D2N for patient 4263333...
Running D2N for patient 7340788...
Running D2N for patient 2351134...
Running D2N for patient 7104512...
Running D2N for patient 6847059...
Running D2N for patient 3541363...
Running D2N for patient 2519384...
Running D2N for patient 2240667...
Running D2N for patient 6160506...
Folder format weird.
Running D2N for patient 0721057...
Chris Rorden's dcm2niiX version v1.0.20241211  GCC13.3.0 x86-64 (64-bit Linux)
Found 61 DICOM file(s)
Convert 61 DICOM as /mnt/1248/open/finley/hydrocephalus/Hydro_1/0721057_20260521/raw/dDTI_b1000_d30_DeIdentified_20260525154338_14 (256x256x50x61)
Conversion required 30.127756 seconds (27.965595 for c

PosixPath('/mnt/1248/open/finley/hydrocephalus/Hydro_1/data.csv')

## Single Chain Run

In [ ]:
# Base
inputpath   : str   = "/mnt/1248/open/finley/test/automate/0662719_20260428"
functions   : list  = [bet, eddy, src, rec, trk, readstats]
silent      : bool  = True

# TRK
trk_label   : str   = "CC"
trk_roi     : list  = ["FreeSurferSeg:CC_Posterior", "FreeSurferSeg:CC_Mid_Posterior", "FreeSurferSeg:CC_Central", "FreeSurferSeg:CC_Mid_Anterior", "FreeSurferSeg:CC_Anterior"]

# READSTATS
read_metrics: list  = ["fa", "md", "ad", "rd", "vol"] # Leave empty to get all metrics

print(chain(
    inputpath       =   inputpath, 
    functions       =   functions, 
    silent          =   silent,
    trk_roi         =   trk_roi, 
    trk_label       =   trk_label, 
    read_metrics    =   read_metrics,
    ))

{'readstats': {'fa': np.float64(0.400992), 'md': np.float64(1.359211), 'ad': np.float64(1.967341), 'rd': np.float64(1.055146), 'vol': np.float64(0.892803)}}
